# Week 8 · Lab Project — Classify Fashion Images with Your Own Network

**This is an assessed lab. You write the model. You write the training loop.**

All week we built neural networks together — a neuron by hand, backprop from scratch, then PyTorch. Now you prove you can do it yourself. This notebook gives you a **real, harder dataset** and does the boring parts for you (loading, preparing, evaluating). The interesting part — **building and training the network** — is yours.

### The dataset: Fashion-MNIST
Instead of handwritten digits, you'll classify **clothing**: 28×28 grayscale photos of shirts, shoes, bags, coats, and more — 10 categories. It looks like the digits problem but it's genuinely harder (a pullover and a coat really do look alike), so your choices about architecture and tuning will actually matter.

### What you have to do
1. ✅ **Data is loaded and prepared for you** (below) — read it, understand it, don't change it.
2. ✍️ **You build the network** — Task A.
3. ✍️ **You write the training loop** — Task B.
4. ✅ **Evaluation is written for you** — run it to see how you did.
5. ✍️ **You tune and write up** — Task C.

> Everything you need is something you did this week. Look back at the Day 5 notebook if you get stuck — but write this yourself.

---
# Part 1 — Setup & Data (provided ✅)

Run these cells as-is. **Read them** — understanding the data is half of any ML project — but you don't need to change anything here.

In [1]:
# torch + torchvision should already be in your conda env (see setup handout)

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
print("torch version:", torch.__version__)

torch version: 2.8.0+cpu


### Load Fashion-MNIST (offline)
The dataset files are already on your machine (your instructor provided them). Each image is 28×28 = **784 pixels**; there are **10 clothing classes**. We take a manageable slice so training is fast in a lab session.

> ⚠️ **Before running:** the folder `./data/FashionMNIST/raw/` must sit next to this notebook, containing the four `*-idx*-ubyte.gz` files from the USB stick. If you get a *FileNotFoundError*, the files aren't in the right place — see the note under the cell.

In [5]:
from torchvision import datasets

# NOTE: download=False — the data is already on disk (provided offline).
# The folder ./data/FashionMNIST/raw/ must contain the four *-ubyte.gz files.
train_raw = datasets.FashionMNIST(root="excercises\data", train=True,  download=False)
test_raw  = datasets.FashionMNIST(root="./data", train=False, download=False)

# the 10 clothing categories
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# take a slice so a lab-session train is quick: 8000 train, 2000 test
train_imgs = train_raw.data[:8000].numpy()
train_lbls = train_raw.targets[:8000].numpy()
test_imgs  = test_raw.data[:2000].numpy()
test_lbls  = test_raw.targets[:2000].numpy()

print("train images:", train_imgs.shape, " test images:", test_imgs.shape)
print("classes:", class_names)

RuntimeError: Dataset not found. You can use download=True to download it

> **If the cell above fails with `FileNotFoundError` or `Dataset not found`:** the dataset isn’t where PyTorch expects it. There must be a `data` folder next to this notebook, arranged exactly like this — note it needs **both** the `.gz` files **and** the extracted (no-extension) files:
>
> ```
> your-notebook.ipynb
> data/
>   FashionMNIST/
>     raw/
>       train-images-idx3-ubyte      train-images-idx3-ubyte.gz
>       train-labels-idx1-ubyte      train-labels-idx1-ubyte.gz
>       t10k-images-idx3-ubyte       t10k-images-idx3-ubyte.gz
>       t10k-labels-idx1-ubyte       t10k-labels-idx1-ubyte.gz
> ```
>
> Copy the whole `data` folder from the USB stick into the same folder as this notebook, then re-run. Don’t rename anything.

### Look at the data first (always)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(train_imgs[i], cmap="gray")
    ax.set_title(class_names[train_lbls[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("Fashion-MNIST — harder than digits (a coat vs a pullover is tricky!)")
plt.tight_layout()
plt.show()

### Prepare the data for the model (provided ✅)
The same honest prep you learned on Day 5, done for you:

1. **Flatten** each 28×28 image into a row of 784 numbers (an MLP takes a flat vector of features).
2. **Scale** pixels from 0–255 down to 0–1 (networks train far better on small, uniform inputs).
3. **To tensors** — floats for the images, longs for the labels.
4. **Wrap in a `DataLoader`** so you get shuffled batches of 64 during training.

After this cell you have everything a model needs: `train_loader`, plus `X_test_t` / `y_test_t` for evaluation.

In [ ]:
# 1. flatten 28x28 -> 784
X_train = train_imgs.reshape(len(train_imgs), -1).astype("float32")
X_test  = test_imgs.reshape(len(test_imgs), -1).astype("float32")

# 2. scale pixels 0-255 -> 0-1
X_train /= 255.0
X_test  /= 255.0

# 3. to tensors
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(train_lbls, dtype=torch.long)
X_test_t  = torch.tensor(X_test)
y_test_t  = torch.tensor(test_lbls, dtype=torch.long)

# 4. batched, shuffled loader for training
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

print("ready for a model:")
print("  each input is a vector of", X_train_t.shape[1], "pixels")
print("  there are", len(class_names), "classes to predict")
print("  train batches per epoch:", len(train_loader))

**The two numbers your model must match:**
- input size = **784** (pixels per image)
- output size = **10** (clothing classes)

Get those two right and the rest is your design. Everything from here is yours to write.

---
# Part 2 — Build Your Network  ✍️ (Task A)

Write a PyTorch model that takes **784 inputs** and produces **10 outputs**.

**Requirements:**
- Input layer must accept **784** features.
- At least **one hidden layer** with a **ReLU** activation (more is fine).
- Output layer must produce **10** values (one score per class). **No softmax** on the end — `CrossEntropyLoss` handles that.

You can write it either style you learned on Day 4 — a class with `nn.Module`, or `nn.Sequential`. Sequential is quickest:

```python
model = nn.Sequential(
    nn.Linear(784, ___),   # <- choose your hidden size
    nn.ReLU(),
    # ... add more layers if you like ...
    nn.Linear(___, 10)     # <- must end at 10
)
```

💡 *Shape rule: the output size of each layer must equal the input size of the next. If PyTorch throws a shape error later, this is almost always why.*

In [ ]:
torch.manual_seed(42)   # keep this so your runs are comparable

# ===== YOUR CODE HERE (Task A) =====
# Build your model. Name it exactly `model` so the rest of the notebook finds it.

model = None   # <-- replace this with your network

# ===================================

print(model)

### Quick self-check (provided ✅)
Run this to confirm your model has the right input/output shape **before** you train. It feeds in 5 fake images and checks the output is `(5, 10)`. Fix Task A until this passes.

In [ ]:
assert model is not None, "Task A: you haven't built `model` yet."
with torch.no_grad():
    dummy = torch.randn(5, 784)
    out = model(dummy)
assert out.shape == (5, 10), f"Wrong output shape {tuple(out.shape)} — it must be (5, 10). Check your first and last layer sizes."
print("✅ shape check passed — your model takes 784 in and gives 10 out. Ready to train.")

---
# Part 3 — Train Your Network  ✍️ (Task B)

Write the training loop. This is the **same four moves** you've written all week: **forward → loss → backward → update**, looped over batches for several epochs.

**Requirements:**
- Use `nn.CrossEntropyLoss()` as the loss (correct choice for multi-class).
- Use an optimizer — `torch.optim.Adam(model.parameters(), lr=0.01)` is a good default.
- Loop over **epochs**, and inside each epoch loop over **batches** from `train_loader`.
- Each batch does the four moves. Don't forget `optimizer.zero_grad()` before `backward()`.
- Append each epoch's loss to `loss_history` so we can plot it.

**Skeleton to fill in:**
```python
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []

for epoch in range(10):
    for xb, yb in train_loader:
        # 1. forward:  preds = model(xb)
        # 2. loss:     loss = loss_fn(preds, yb)
        # 3. backward: zero_grad, then loss.backward()
        # 4. update:   optimizer.step()
        pass
    loss_history.append(loss.item())   # last batch loss of the epoch
    print(f"epoch {epoch+1}: loss {loss.item():.4f}")
```

💡 *Start with ~10 epochs. You can raise it later in Task C.*

In [ ]:
# ===== YOUR CODE HERE (Task B) =====
# Write the training loop. Keep a list called `loss_history` for the plot below.

loss_history = []


# ===================================


### Plot your training loss (provided ✅)
If your loop worked, this curve should fall. A falling loss means your network is learning.

In [ ]:
assert len(loss_history) > 0, "Task B: loss_history is empty — did the training loop run and append losses?"
plt.plot(loss_history, marker="o", color="purple")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Your network learning (loss should fall)")
plt.grid(alpha=0.3)
plt.show()

---
# Part 4 — Evaluate (provided ✅)

You built it and trained it — now let's see how well it does. This section is written for you. Run it to get your **test accuracy**, a **confusion matrix**, and a look at some predictions. **Read the results** — you'll need them for the write-up.

In [ ]:
# overall test accuracy
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    preds = logits.argmax(dim=1)

accuracy = (preds == y_test_t).float().mean().item()
print(f"🎯 TEST ACCURACY: {accuracy:.2%}")
print("(A simple MLP typically lands around 82-88% here. Above 88% is strong.)")

In [ ]:
# confusion matrix — which clothes get mixed up?
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_t, preds)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"Confusion matrix — {accuracy:.1%} accuracy")
plt.tight_layout()
plt.show()

In [ ]:
# per-class accuracy — what is your network good and bad at?
print("class-by-class accuracy:\n")
for c in range(10):
    mask = (y_test_t == c)
    if mask.sum() > 0:
        acc_c = (preds[mask] == c).float().mean().item()
        bar = "█" * int(acc_c * 20)
        print(f"  {class_names[c]:<12} {acc_c:5.1%}  {bar}")

In [ ]:
# look at some predictions — green = correct, red = wrong
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for i, ax in enumerate(axes.ravel()):
    img = test_imgs[i]
    true_c, pred_c = y_test_t[i].item(), preds[i].item()
    ax.imshow(img, cmap="gray")
    color = "green" if true_c == pred_c else "red"
    ax.set_title(f"pred: {class_names[pred_c]}\ntrue: {class_names[true_c]}", fontsize=8, color=color)
    ax.axis("off")
plt.suptitle("Your network's predictions (green = right, red = wrong)")
plt.tight_layout()
plt.show()

---
# Part 5 — Tune & Write Up  ✍️ (Task C)

Your first model works. Now **improve it**, and report what you found. This is where the marks are.

### Step 1 — Try to beat your score
Go back to **Task A / Task B** and change **one thing at a time**, re-running to see the effect. Ideas:
- a **bigger hidden layer**, or an **extra** hidden layer
- a different **learning rate** (try 0.001 and 0.1)
- **more epochs** (try 20–30)
- add `nn.Dropout(0.2)` between layers to fight overfitting

Keep track of what each change did to your test accuracy.

### Step 2 — Write it up
Double-click this cell and fill in your answers:

**My best test accuracy:** ____%

**My final network (layers and sizes):**
> _e.g. 784 → 256 → 128 → 10, ReLU, dropout 0.2_

**What I changed and what happened** (at least two experiments):
> 1. 
> 2. 

**Which two clothing classes did my network confuse most?** (look at the confusion matrix)
> 

**One thing that surprised me:**
> 

---
## Submission checklist

Before you push to GitHub, confirm:

- [ ] **Task A** — model built, self-check passed (784 in → 10 out).
- [ ] **Task B** — training loop written, loss curve falls.
- [ ] **Part 4** — evaluation run; you have an accuracy number and confusion matrix.
- [ ] **Task C** — at least two tuning experiments done, write-up complete.
- [ ] Notebook **runs top to bottom** without errors (Kernel → Restart & Run All).

**Push to:** `week-08/neural-network-project.ipynb`

---
### How you're graded
| | |
|---|---|
| Model built correctly (right shapes, ReLU, 10 outputs) | 25% |
| Training loop correct (four moves, loss falls) | 30% |
| Evaluation read and understood (write-up references real results) | 20% |
| Tuning experiments (at least two, with observations) | 15% |
| Notebook runs cleanly top to bottom | 10% |

*You built a neural network that recognizes clothing from raw pixels — from scratch, in the framework professionals use. That's a real skill. Well done.* 👏